# Evaluating Agentic RAG: Breaking and Fixing a Claims Assistant

In the first half of the lab, we used LangGraph to extract structured data from messy adjuster notes. Now, we flip the script: we are going to build a Retrieval-Augmented Generation (RAG) agent that answers adjuster questions using the **06-2025 FEMA Claims Manual**, and then we are going to rigorously evaluate its performance using Snowflake's native `CORTEX.EVALUATE` functions.

## Learning Objectives

1. Ingest and chunk a massive PDF manual into a **Cortex Search Service**
2. Build a baseline RAG prompt (and watch it fail on complex claims)
3. Upgrade to an Agentic workflow to improve retrieval accuracy
4. Programmatically measure groundedness, relevance, and hallucinations natively in Snowflake

---
# Section 1: Building the Knowledge Base

Before we can evaluate an agent, it needs something to read. The [06-2025 FEMA Claims Manual] is hundreds of pages of dense policy rules. 

**Where you are in the overall pipeline:**

```text
[ FEMA PDF ] → [ LangChain Splitter ] → [ Cortex Search Service ] → Agent
                       ↑ YOU ARE HERE

In [ ]:
### Cell 3 (Python Code)
import json
import time
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.files import SnowflakeFile
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Connect to our Snowflake Session
session = get_active_session()
print(f"Connected to Snowflake: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema: {session.get_current_schema()}")
print(f"Warehouse: {session.get_current_warehouse()}")

In [ ]:
# 2. Extract and Chunk the PDF
# We use LangChain's recursive splitter to keep paragraphs together 
# while staying under the LLM context window limits.

# Note: In a true production environment, you might use Snowflake's native 
# CORTEX.PARSE_DOCUMENT or pdfplumber, but PyPDF/LangChain works great for this lab!
from langchain_community.document_loaders import PyPDFLoader

# We download the file from the stage to the notebook's ephemeral storage to parse it
stage_file_path = "@claims_extraction_repo/branches/main/06-2025-claims-manual.pdf"
local_file_path = "/tmp/06-2025-claims-manual.pdf"

session.file.get(stage_file_path, "/tmp/")

loader = PyPDFLoader(local_file_path)
pages = loader.load()
print(f"Loaded {len(pages)} pages from the Claims Manual.")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(pages)
print(f"Split into {len(chunks)} searchable chunks.")

# 3. Save chunks to a Snowflake Table
# We extract the text and metadata into a list of dictionaries, then convert to a Snowpark DataFrame
chunk_data = [
    {
        "CHUNK_ID": f"chunk_{i}", 
        "PAGE_NUM": chunk.metadata.get("page", 0),
        "CHUNK_TEXT": chunk.page_content
    } 
    for i, chunk in enumerate(chunks)
]

df_chunks = session.create_dataframe(chunk_data)
df_chunks.write.mode("overwrite").save_as_table("CLAIMS_MANUAL_CHUNKS")
print("Successfully saved chunks to CLAIMS_MANUAL_CHUNKS table.")

In [ ]:
# 4. Create the Cortex Search Service
# This one SQL command handles text embedding, vector database provisioning, and similarity search indexing!

current_wh = session.get_current_warehouse()

if not current_wh:
    raise ValueError("No warehouse selected! Please select a warehouse in the top right of your Snowpark Notebook.")

# Notice the 'f' before the triple quotes to enable string interpolation
create_css_sql = f"""
CREATE OR REPLACE CORTEX SEARCH SERVICE claims_manual_search_svc
  ON chunk_text
  ATTRIBUTES page_num
  WAREHOUSE = {current_wh}
  TARGET_LAG = '1 minute'
  AS (
    SELECT chunk_id, page_num, chunk_text
    FROM CLAIMS_MANUAL_CHUNKS
  );
"""

session.sql(create_css_sql).collect()
print(f"Cortex Search Service 'claims_manual_search_svc' is live and indexing using {current_wh}!")

> **Takeaway:** By defining `ATTRIBUTES page_num` in our Cortex Search Service, we are allowing our future Agent to pre-filter its vector searches (e.g., "Only search for this rule between pages 50 and 60"). This is a critical pattern for improving RAG accuracy on massive documents!

---
# Section 2: The "Naive" RAG Trap

Now that our FEMA Claims Manual is indexed in Cortex, let's build a basic RAG (Retrieval-Augmented Generation) pipeline. 

The standard approach most developers take on day one is:
1. Take the user's question.
2. Search the vector database for relevant chunks.
3. Stuff those chunks into a massive prompt.
4. Ask the LLM for an answer.

Let's see what happens when we apply this "Naive RAG" approach to a complicated flood claim. Flood insurance rules around **basements** and **enclosures** are notoriously strict—NFIP severely limits what contents are covered below the lowest elevated floor. 

Let's query our `FIMA_CLAIMS` table for a claim that might fall into this trap.

In [ ]:
# 1. Find a "Tricky" Claim 
# We are looking for a claim that has a basement (basement_type > 0) 
# and a high amount of contents damage claimed.

tricky_claim_df = session.sql("""
    SELECT 
        claim_id, 
        state,
        basement_type, 
        contents_damage_amount, 
        building_damage_amount,
        adjuster_notes
    FROM FIMA_CLAIMS
    WHERE basement_type IN ('1', '2', '3', '4') -- Codes for basements/enclosures
      AND contents_damage_amount > 15000
    LIMIT 1
""").collect()

claim = tricky_claim_df[0]

print(f"Target Claim ID: {claim['CLAIM_ID']}")
print(f"Basement Type: {claim['BASEMENT_TYPE']}")
print(f"Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']:,.2f}")
print(f"\nAdjuster Notes:\n{claim['ADJUSTER_NOTES']}")

### Querying the Cortex Search Service

Next, we will query our Cortex Search Service for rules regarding basement contents coverage. We'll use the `snowflake.core` Python API (the modern standard for interacting with Snowflake services natively) to pull the top 5 most relevant chunks from the manual.

In [ ]:
import pandas as pd
from snowflake.core import Root

# Initialize the Snowflake Core Root object
root = Root(session)
db = session.get_current_database()
sch = session.get_current_schema()

# Connect to our Search Service
search_svc = root.databases[db].schemas[sch].cortex_search_services["claims_manual_search_svc"]

# 2. Perform the Vector Search
search_query = "What personal property or contents are covered in a basement or crawlspace?"

# We ask for the top 5 chunks
search_results = search_svc.search(
    query=search_query,
    columns=["CHUNK_TEXT", "PAGE_NUM"],
    limit=5
)

# Extract the text chunks to use as context
retrieved_chunks = [row["CHUNK_TEXT"] for row in search_results.results]

print(f"Retrieved {len(retrieved_chunks)} chunks from the FEMA manual.")
print("\nPreview of top chunk:")
print("-" * 40)
print(retrieved_chunks[0][:300] + "...")

### The Naive RAG Prompt

Now we stuff everything—the claim details and the retrieved manual chunks—into a single prompt for `claude-3-5-sonnet`. We are asking the model to act as a claims advisor and determine if we should pay out the contents damage.

In [ ]:
# 3. Build the context string
context_str = "\n\n---\n\n".join(retrieved_chunks)

# 4. Construct the Prompt
system_prompt = """
You are an expert FEMA NFIP Flood Insurance Advisor. 
Review the provided claim details and use ONLY the provided FEMA Claims Manual excerpts to advise the adjuster.

Determine if the claimed contents damage should be paid out in full, partially, or denied based on the manual's rules regarding basements.
"""

user_prompt = f"""
CLAIM DETAILS:
- Claim ID: {claim['CLAIM_ID']}
- Basement Type Code: {claim['BASEMENT_TYPE']} (Has a basement/enclosure)
- Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']}
- Adjuster Notes: {claim['ADJUSTER_NOTES']}

FEMA MANUAL EXCERPTS:
{context_str}

Based on the manual excerpts, how should we handle the $15,000+ contents damage in the basement for this claim? Be definitive.
"""

# 5. Call Cortex Complete - Note the updated SQL to extract just the string message!
rag_prompt = json.dumps([
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
])

naive_response = session.sql(f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${rag_prompt}$$),
        {{'temperature': 0.1}}
    ):choices[0]:messages::STRING AS agent_response
""").collect()[0][0]

print("🤖 NAIVE RAG AGENT RESPONSE:")
print("=============================")
print(naive_response)

> **Takeaway:** Read the LLM's response carefully. Did it definitively list *exactly* what is covered (like washers, dryers, and freezers) and explicitly deny general personal property (like TVs and couches) which are excluded in basements under NFIP rules? 
>
> Often, a Naive RAG approach will confidently hallucinate or provide a vague "pay it" response because the basic text chunking cut the exclusion list in half, or the LLM lost the strict policy rule in the noise of the prompt. 

This is exactly why we cannot push AI into production without **Evaluation**. In the next section, we will use Snowflake's native evaluation tools to mathematically prove that this agent's response is lacking, and then we will build a better one.

---
### Architectural Review: Why Naive RAG Fails

Before we evaluate, let's look at the pipeline we just built. In **Naive RAG**, the architecture is strictly linear. The LLM has no agency, no ability to double-check its work, and is entirely at the mercy of the vector search's initial accuracy.

```text
[ User Query ] 
      │
      ▼
╭────────────╮      (Top-4 Chunks)      ╭──────────────╮
│ Vector DB  │ ───────────────────────► │   Naive LLM  │
╰────────────╯                          ╰──────┬───────╯
(Semantic Search Gap)                          │
                                               ▼
                                   [ Hallucinated Answer ]
```

Because of the **Semantic Search Gap** (where the database retrieved chunks about "basement limitations" but missed the specific chunk listing "washers/dryers"), the Naive LLM was forced to guess. 

To prove this programmatically, we will build an **Oracle Judge**.

---
# Section 3: Programmatic Evaluation (The Oracle Judge)

In the real world, you cannot manually read every RAG response, and building a massive database of "Ground Truth" strings for every possible claim scenario is impossible. 

Instead, we use a pattern called **LLM-as-a-Judge**. 

Because the FEMA NFIP manual is a public federal document, frontier models (like Claude 3.5 Sonnet) already possess deep foundational knowledge of its rules from their initial pre-training. We are going to leverage that! We will spin up a second Cortex call acting as a "Master Adjuster." We will ask it to look at the Naive Agent's answer and grade it based on its *own* internal knowledge of the true policy.

We want scores (0-5) on two metrics:
1. **Context Sufficiency:** Did our naive vector search actually pull the right rules?
2. **Answer Correctness:** Did the naive agent give an answer that aligns with actual FEMA policy?

In [ ]:
import json
import textwrap

# 1. Build the Dynamic Evaluation Prompt
# We rely on the "Oracle" model's pre-trained expertise
eval_prompt = f"""
You are a Master FEMA NFIP Claims Adjuster and an impartial Evaluation Judge.
You have comprehensive, expert knowledge of the official FEMA NFIP Claims Manual, including the strict and specific rules regarding coverage limitations in basements and enclosures.

A junior automated Agent was asked to review a claim. It was provided with a limited set of Retrieved Context and generated an Answer.

CLAIM DETAILS:
- Basement Type: {claim['BASEMENT_TYPE']}
- Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']}

RETRIEVED CONTEXT PROVIDED TO JUNIOR AGENT: 
{context_str}

JUNIOR AGENT'S ANSWER:
{naive_response}

Based on your expert, foundational knowledge of the ACTUAL FEMA NFIP manual, evaluate the junior agent's performance. 
"""

# 2. Package the prompt
judge_payload = json.dumps([
    {"role": "system", "content": "You are a JSON-outputting evaluation system."},
    {"role": "user", "content": eval_prompt}
])

# 3. Define the exact JSON Schema for the Judge (Structured Output)
eval_schema = {
    "type": "object",
    "properties": {
        "context_score": {
            "type": "integer",
            "description": "Score from 0 to 5. Did the RETRIEVED CONTEXT contain the necessary rules?"
        },
        "correctness_score": {
            "type": "integer",
            "description": "Score from 0 to 5. Is the JUNIOR AGENT'S ANSWER factually correct according to true FEMA policy?"
        },
        "explanation": {
            "type": "string",
            "description": "A 2-sentence explanation of what the context missed and what the agent got wrong."
        }
    },
    "required": ["context_score", "correctness_score", "explanation"]
}

# 4. Pass the schema into the options
judge_options = json.dumps({
    "temperature": 0,
    "response_format": {
        "type": "json",
        "schema": eval_schema
    }
})

# 5. Use Cortex to Grade the Response
judge_query = f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${judge_payload}$$),
        PARSE_JSON($${judge_options}$$)
    ):structured_output[0]:raw_message::STRING AS evaluation
"""

# Execute the judge
eval_result_str = session.sql(judge_query).collect()[0][0]

# Now it will correctly parse!
eval_result = json.loads(eval_result_str)

explanation = eval_result.get('explanation', '')
wrapped_explanation = textwrap.fill(explanation, width=80)

print("⚖️ ORACLE EVALUATION RESULTS:")
print("=============================")
print(f"Context Sufficiency Score : {eval_result.get('context_score')}/5")
print(f"Answer Correctness Score  : {eval_result.get('correctness_score')}/5")
print("Explanation               :")
print(wrapped_explanation)

> **Takeaway:** The Judge correctly flagged our Naive RAG pipeline! 
> 
> The **Context Score** is low because our basic 1,000-character chunking strategy and Top-5 vector search completely missed the section of the manual detailing washer/dryer exceptions. 
> 
> The **Correctness Score** is low because the agent, lacking the right context, hallucinated a blanket denial. 

If this were running across 10,000 claims in production, these metrics would immediately alert your engineering team that the retrieval pipeline is failing for basement-related claims.

---
### Architectural Review: Advanced Agentic RAG (ReAct + Referee)

To fix the pipeline, we are abandoning the linear Naive RAG approach and building an **Agentic Orchestration Loop**. 

This architecture uses the **ReAct (Reasoning and Acting)** framework. Instead of just answering, the Main Agent is given a "Tool" and forced into a loop where it must Think, Act, and Observe before answering. 

To prevent the "Lost in the Middle" problem (where the Agent gets overwhelmed by too many search results), we are injecting a **Referee Agent** into the tool itself to filter the noise.

```text
       [ Claim Details ]
               │
               ▼
      ╭─────────────────────╮
      │     MAIN AGENT      │ ◄─────────────────────────┐
      │  (Claude 3.5 Sonnet)│                           │
      ╰────────┬────────────╯                           │
               │                                        │
        [Action: 'search']                              │
               │                                        │
               ▼                                        │
      ╭─────────────────────╮                           │
      │    SEARCH TOOL      │                           │
      │  (Pulls 20 Chunks)  │                           │
      ╰────────┬────────────╯                           │
               │                                        │
               ▼                                        │
      ╭─────────────────────╮                           │
      │   REFEREE AGENT     │                           │
      │  (Filters noise &   │ ──(Yields Exact Rules)────┘
      │  extracts citations)│          
      ╰─────────────────────╯        
               │
        [Action: 'answer']
               │
               ▼
    [ Factually Correct Answer ]
```

Notice that we are not using a heavy orchestration framework like LangGraph or CrewAI here. We are building this loop entirely from scratch using native Python and Snowflake SQL so you can see exactly how Agentic routing works under the hood!

---
# Section 4: Fixing the Pipeline (Agentic RAG)

Why did we use the public FEMA Claims Manual for this lab instead of a proprietary, internal insurance dataset? 

**Because we needed an Oracle.** In the real world, when you deploy RAG on your company's proprietary knowledge base, there is no pre-trained Oracle that knows the ground truth. If the model hallucinates, you might not catch it. Therefore, you must validate your RAG *architecture* on data where a ground-truth Oracle *can* exist. 

By testing our methods here, we proved that Naive RAG is dangerously flawed (scoring 2/5 on Context and 4/5 on Correctness). Now, we will upgrade to an **Agentic RAG** architecture. Once we mathematically prove that this new architecture can score a perfect 5/5 on our test data, **we can trust the pipeline to become the Oracle for our proprietary data.**

### The Fix: Tool Calling
Instead of doing a single, blind vector search and stuffing the prompt, we will give the LLM a **Tool**. The Agent will read the claim first, formulate a highly specific search query (like *"are washers and dryers covered in a basement?"*), and pull exactly what it needs before answering.

In [ ]:
import json
import textwrap

# 1. Define the "Referee" Search Tool
def search_fema_manual_advanced(query: str) -> str:
    print(f"\n    [Agent Tool Execution] 🔍 Searching KB for: '{query}'")
    
    # Step A: Pull a wide net of chunks (Top 20)
    results = search_svc.search(
        query=query,
        columns=["CHUNK_TEXT"],
        limit=20
    )
    raw_chunks = "\n\n---\n\n".join([row["CHUNK_TEXT"] for row in results.results])

    print(f"    [Referee Agent] 🧐 Scanning {len(results.results)} chunks to filter noise...")
    
    # Step B: The Referee LLM filters the noise    
    referee_prompt = f"""
    You are a strict text-extraction referee. 
    Review the following raw text chunks from the FEMA manual. 
    Find ANY mention of exceptions, coverage allowances, or specific rules relating to this query: "{query}".
    
    CRITICAL: If you find rules about items being COVERED or EXCLUDED, extract them directly AND include any section numbers, policy citations (like SFIP III.B.3), or headers you see near the text. 
    Ignore all irrelevant text. Do not summarize, just extract the hard facts and their citations.
    If the text contains nothing relevant, reply with "No relevant rules found."
    
    RAW CHUNKS:
    {raw_chunks}
    """
    
    # Package the prompt into a standard JSON payload
    referee_payload = json.dumps([
        {"role": "user", "content": referee_prompt}
    ])
    
    referee_options = json.dumps({"temperature": 0})
    
    # Use PARSE_JSON to force the compiler to accept the types
    referee_query = """
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON(?),
            PARSE_JSON(?)
        ):choices[0]:messages::STRING
    """
    
    # Execute the Referee
    filtered_context = session.sql(
        referee_query, 
        params=[referee_payload, referee_options]
    ).collect()[0][0]
    
    print(f"    [Referee Agent] ✅ Condensed facts:")
    # Print the actual filtered facts indented nicely!
    print(textwrap.indent(textwrap.fill(filtered_context, width=76), "        "))
    
    return filtered_context
# 2. Define the Agent's JSON Schema
agent_schema = {
    "type": "object",
    "properties": {
        "thought_process": {
            "type": "string",
            "description": "Your internal reasoning. Explain what information you need."
        },
        "action": {
            "type": "string",
            "description": "Must be EXACTLY 'search' or 'answer'."
        },
        "search_query": {
            "type": "string",
            "description": "If action is 'search', the specific keywords to look up."
        },
        "final_recommendation": {
            "type": "string",
            "description": "If action is 'answer', your definitive policy recommendation."
        }
    },
    "required": ["thought_process", "action"]
}

agent_options = json.dumps({
    "temperature": 0.1,
    "response_format": {"type": "json", "schema": agent_schema}
})

def run_agent_step(msgs):
    query = """
        SELECT SNOWFLAKE.CORTEX.COMPLETE(
            'claude-3-5-sonnet',
            PARSE_JSON(?),
            PARSE_JSON(?)
        ):structured_output[0]:raw_message::STRING
    """
    raw_result = session.sql(query, params=[json.dumps(msgs), agent_options]).collect()[0][0]
    return json.loads(raw_result)

print("✅ Advanced Referee Search Tool defined!")

In [ ]:
# 4. Initialize the Agent's Memory
agent_system_prompt = """
You are a precise FEMA NFIP Claims Agent. Review the claim details.
DO NOT guess the policy rules. You must use the 'search' action to look up specific items (e.g., washers, dryers, furniture).
CRITICAL INSTRUCTION: Once you have executed 1 or 2 searches and retrieved the policy rules, you MUST output the 'answer' action. Do not search endlessly. If the text says items are excluded, accept it and output your final recommendation.
"""

agent_user_prompt = f"""
CLAIM DETAILS:
- Basement Type: {claim['BASEMENT_TYPE']} (Has a basement/enclosure)
- Contents Claimed: Washer/dryer, shelving units, area rugs, stored goods.
- Claim Amount: ${claim['CONTENTS_DAMAGE_AMOUNT']}
"""

messages = [
    {"role": "system", "content": agent_system_prompt},
    {"role": "user", "content": agent_user_prompt}
]

print("🤖 ADVANCED AGENTIC RAG INITIALIZED...")

# 5. The Execution Loop (Bumped to Max 5 steps)
for step in range(5):
    print(f"\n--- Agent Step {step + 1} ---")
    
    # Let the Agent think and act
    response = run_agent_step(messages)
    
    # Print the Agent's internal monologue
    print("🧠 Thought:", textwrap.fill(response.get("thought_process", ""), width=80))
    
    action = response.get("action")
    
    if action == "search":
        query = response.get("search_query", "basement coverage")
        search_results = search_fema_manual_advanced(query)
        
        # Add the interaction to the agent's memory
        messages.append({"role": "assistant", "content": json.dumps(response)})
        messages.append({"role": "user", "content": f"TOOL RESULTS:\n{search_results}"})
        
    elif action == "answer":
        print("\n✅ FINAL AGENT RECOMMENDATION:")
        print("==============================")
        print(textwrap.fill(response.get("final_recommendation", ""), width=80))
        
        # Save the answer for the Oracle Judge
        agentic_answer = response.get("final_recommendation", "")
        search_result_text = "\n".join([m["content"] for m in messages if m["role"] == "user"])
        break
else:
    print("Agent timed out before reaching an answer.")

> **Takeaway:** Look at the print statements above! The Agent read the claim, realized it needed specific information about washers and dryers in basements, and generated a targeted search query. Because the search was highly specific, it successfully retrieved the SFIP exceptions. 
>
> Let's prove it by sending this new pipeline back to our Oracle Judge.

In [ ]:
# 1. Build the Dynamic Evaluation Prompt using the NEW Agent's answer and context
eval_prompt = f"""
You are a Master FEMA NFIP Claims Adjuster and an impartial Evaluation Judge.
You have comprehensive, expert knowledge of the official FEMA NFIP Claims Manual, including the strict and specific rules regarding coverage limitations in basements and enclosures.

An upgraded Agentic RAG system was asked to review a claim. 

CLAIM DETAILS:
- Basement Type: {claim['BASEMENT_TYPE']}
- Contents Damage Claimed: ${claim['CONTENTS_DAMAGE_AMOUNT']}

RETRIEVED CONTEXT PROVIDED TO THE AGENT: 
{search_result_text}

AGENT'S ANSWER:
{agentic_answer}

Based on your expert, foundational knowledge of the ACTUAL FEMA NFIP manual, evaluate the agent's performance. 
"""

# 2. Package the prompt
judge_payload = json.dumps([
    {"role": "system", "content": "You are a JSON-outputting evaluation system."},
    {"role": "user", "content": eval_prompt}
])

# 3. Use Cortex to Grade the Response
judge_query = f"""
    SELECT SNOWFLAKE.CORTEX.COMPLETE(
        'claude-3-5-sonnet',
        PARSE_JSON($${judge_payload}$$),
        PARSE_JSON($${judge_options}$$)
    ):structured_output[0]:raw_message::STRING AS evaluation
"""

# Execute the judge
eval_result_str = session.sql(judge_query).collect()[0][0]
eval_result = json.loads(eval_result_str)

print("⚖️ FINAL ORACLE EVALUATION RESULTS:")
print("===================================")
print(f"Context Sufficiency Score : {eval_result.get('context_score')}/5")
print(f"Answer Correctness Score  : {eval_result.get('correctness_score')}/5")
print("Explanation               :")
print(textwrap.fill(eval_result.get('explanation', ''), width=80))

---
# Conclusion: From Lab to Enterprise with LangGraph

Throughout this lab, we experienced the realities of AI engineering. We saw how a naive vector search confidently hallucinated a compliance violation. We saw how bad chunking strategies hide critical policy exceptions. And finally, we saw how **Agentic Architecture** (using tools, Referees, and ReAct loops) can overcome those hurdles to find the exact SFIP citations needed for a correct decision.

### The Missing Piece: Automated Validation
In this lab, we manually tweaked prompts and top-k limits to get our Agent to pass the Oracle Judge for a *single* claim. In an enterprise environment, you cannot manually read every log.

To deploy this securely on your proprietary data, the next step is to build a **LangGraph Evaluation Harness**:
1. **The Golden Dataset:** Curate a sample batch of 50-100 tricky claims representing different edge cases (basements, elevated buildings, mold, etc.).
2. **Parallel Execution:** Use LangGraph to run those 100 claims through your Agentic Pipeline.
3. **The Automated Referee:** Use the Cortex Oracle Judge (from Section 3) to programmatically score every single response for Context Sufficiency and Correctness.
4. **Iterative Tuning:** Adjust your chunking strategies (e.g., using Snowflake's Document AI for layout-aware parsing) or Agent prompts, and re-run the LangGraph evaluator. 

**The ultimate goal is to mathematically prove your architecture.** Once your LangGraph evaluation harness consistently outputs 5/5 scores on your Golden Dataset, the pipeline itself *becomes* the Oracle. You can then safely deploy it against hundreds of thousands of claims, trusting that the architecture will hold up in production.

***End of Lab***